## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [ ]:
!pip install kagglehub
!pip install nltk scikit-learn rank-bm25 pandas

import kagglehub
import pandas as pd
import nltk
import re
import os
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Descargar recursos de NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# Descargar el dataset
path = kagglehub.dataset_download("gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects")
print("Path to dataset files:", path)

# Cargar el archivo CSV (asumiendo que hay un archivo CSV en el dataset)
# Primero, listamos los archivos
files = os.listdir(path)
print("Archivos disponibles:", files)

# Buscar archivo CSV
csv_files = [f for f in files if f.endswith('.csv')]
if csv_files:
    df = pd.read_csv(os.path.join(path, csv_files[0]))
    print(f"Dataset cargado con {len(df)} documentos")
    print(df.head())
else:
    print("No se encontró archivo CSV. Verificando otros formatos...")
    # Si hay JSON u otro formato, ajustar aquí
    print("Contenido del directorio:", files)

## Procesar documentos

In [1]:
# Preprocesamiento del corpus
def preprocess_text(text):
    """
    Función para preprocesar texto:
    - Convertir a minúsculas
    - Eliminar caracteres especiales y números
    - Tokenizar
    - Eliminar stopwords
    - Aplicar stemming
    """
    if not isinstance(text, str):
        return ""
    
    # Convertir a minúsculas
    text = text.lower()
    
    # Eliminar caracteres especiales y números (solo letras y espacios)
    text = re.sub(r'[^a-záéíóúñ\s]', '', text)
    
    # Tokenizar
    tokens = word_tokenize(text, language='spanish')
    
    # Cargar stopwords en español
    stop_words = set(stopwords.words('spanish'))
    
    # Stemmer
    stemmer = PorterStemmer()
    
    # Filtrar stopwords y aplicar stemming
    tokens = [stemmer.stem(token) for token in tokens if token not in stop_words and len(token) > 2]
    
    return ' '.join(tokens)



## Aplicar procesamiento a cada texto

In [ ]:
import pandas as pd
# Aplicar preprocesamiento a los documentos
text_column = None
for col in df.columns:
    if 'text' in col.lower() or 'content' in col.lower() or 'article' in col.lower():
        text_column = col
        break

if text_column:
    print(f"Procesando columna: {text_column}")
    df['processed_text'] = df[text_column].apply(preprocess_text)
    # Eliminar documentos vacíos
    df = df[df['processed_text'].str.strip() != '']
    print(f"Documentos después de preprocesamiento: {len(df)}")
else:
    print("No se encontró columna de texto. Usando primera columna de tipo string")
    # Buscar primera columna de tipo string
    for col in df.columns:
        if df[col].dtype == 'object':
            text_column = col
            break
    df['processed_text'] = df[text_column].apply(preprocess_text)
    df = df[df['processed_text'].str.strip() != '']